In [2]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

import pandas as pd

# pd.set_option('display.max_columns', None)  # none은 제한값 자리(몇개 열까지 보여줄까)
# pd.set_option('display.max_rows', None)

df = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / 'train.parquet')

# 해당 df의 메모리사용량 확인용
# df.memory_usage(deep=True).sum() / 1024**3

In [2]:
# 컬럼 분리하기

# 시계열 컬럼군
time_cols = ['timestamp', # 필요해서 추가함
        'wl_t',          # 수위데이터 대표
        'sfc_rain_1h',   # 육상 누적강수데이터 대표
        'imerg_rain_1h', # 레이더 누적강수데이터 대표
        'radar_reflectivity_mean', # 레이더반사도 대표
        'radar_reflectivity_p95', # 레이더반사도 대표
        'target_maxrise_6h' # 종속변수: 수위상승량
]

# 수위 컬럼군
wl_cols = ['wl_t',
          'wl_t_minus_1h',
          'wl_t_minus_3h',
          'wl_t_minus_6h',
          'wl_t_minus_12h',
          'wl_t_minus_24h',
          'target_maxrise_6h',
]

# 강수 컬럼군
rain_cols = [
            # IMERG강우
            'imerg_rain_1h',
            'imerg_rain_6h',
            'imerg_rain_24h',
            # 지상강우
            'sfc_rain_1h',
            'sfc_rain_6h',
            'sfc_rain_24h',
            # 레이더 반사도
            'radar_reflectivity_mean',
            'radar_reflectivity_p95',
            'radar_reflectivity_ge20_fraction'
]

# 환경 컬럼군
env_cols = [
    'cat_area',       # km²
    'wsarea',         # km²
    'mean_elevation', # m
    'mean_slope',     # °
    'urban_ratio',    # %
    'forest_ratio',   # %
    'stream_order'
]

In [3]:
# 시계열데이터 중에 수위변동값('target_maxrise_6h')에 이상치가 의심됨
cond_min_maxrise = df['target_maxrise_6h'] < -2  # 이거 자체가 불리언 조건문임
print(f"6h이내 수위변동이 -2보다 작은 데이터 개수: {cond_min_maxrise.sum()}")  # 불리언조건문은 sum으로 개수를 셀 수 있음
print()
print("<위 조건에 해당하는 데이터>")
print()
display(df[cond_min_maxrise])
# 수위변동값이 -2 이하인 케이스를 보면 이상치 혼자 다른 관측소 값임
# 이 이상치는 심지어 극한 홍수가 발생한 케이스임에도 수위가 빠졌음
# 해당 관측소값만 모아볼 필요가 있음
print()


# 관측소번호 'ST0007'만 모아보기
cond_ST0007 = df['station_id'] == 'ST0007'
# df.loc[cond_ST0007].sort_values('target_maxrise_6h')
# 수위변동 -8.7은 독보적인 값이 맞음, 주변값들을 볼 필요가 있을듯

# timestamp 기준으로 2103-02-26 00:00 ~ 2103-02-29 00:00 을 모아보겠다.
cond_time = (df['timestamp'] > '2103-02-27 00:00') & (df['timestamp'] < '2103-03-01 00:00')
display(df.loc[cond_ST0007 & cond_time, time_cols].sort_values('timestamp'))
# 이상치 확정
print()
df['target_maxrise_6h'].idxmin()

6h이내 수위변동이 -2보다 작은 데이터 개수: 6

<위 조건에 해당하는 데이터>



,timestamp,station_id,wl_t,wl_t_minus_1h,wl_t_minus_3h,wl_t_minus_6h,wl_t_minus_12h,wl_t_minus_24h,imerg_rain_1h,imerg_rain_6h,...,radar_reflectivity_p95,radar_reflectivity_ge20_fraction,cat_area,wsarea,mean_elevation,mean_slope,urban_ratio,forest_ratio,stream_order,target_maxrise_6h
row_id,,,,,,,,,,,,,,,,,,,,,
TR_000293053,2101-03-07 01:00:00,ST0358,1.34,3.25,5.21,-0.18,0.87,0.00,0.000,0.000,...,11.140,0.0049,6.511,515.2043,28.46,5.1067,14.6533,17.4404,2.3333,-2.18
TR_000706035,2102-03-03 01:00:00,ST0358,1.97,3.21,3.82,-0.47,1.60,1.19,2.336,4.973,...,35.700,0.9961,6.511,515.2043,28.46,5.1067,14.6533,17.4404,2.3333,-2.10
TR_000730464,2102-04-01 09:00:00,ST0358,1.95,3.53,4.57,-0.58,2.17,0.78,0.000,41.369,...,23.581,0.1523,6.511,515.2043,28.46,5.1067,14.6533,17.4404,2.3333,-2.09
TR_000583385,2103-02-27 05:00:00,ST0007,8.75,7.40,4.53,2.31,2.37,2.50,14.154,98.900,...,33.550,0.9883,2.986,99.0096,335.78,16.2836,1.0168,87.6109,2.0588,-8.74
TR_000482942,2103-05-06 06:00:00,ST0358,1.99,3.43,4.36,0.51,-0.16,0.40,2.467,10.497,...,38.144,1.0000,6.511,515.2043,28.46,5.1067,14.6533,17.4404,2.3333,-2.07
TR_000356824,2103-07-15 03:00:00,ST0358,3.11,4.58,4.81,-1.30,1.36,1.69,0.640,0.727,...,48.770,0.5225,6.511,515.2043,28.46,5.1067,14.6533,17.4404,2.3333,-2.12


,timestamp,wl_t,sfc_rain_1h,imerg_rain_1h,radar_reflectivity_mean,radar_reflectivity_p95,target_maxrise_6h
row_id,,,,,,,
TR_000459786,2103-02-27 01:00:00,2.96,56.5,4.423,48.456,52.375,5.79
TR_000121377,2103-02-27 02:00:00,4.53,40.0,17.038,37.736,44.261,4.22
TR_000128421,2103-02-27 03:00:00,5.35,34.0,25.780,46.048,54.260,3.40
TR_000397383,2103-02-27 04:00:00,7.40,79.5,28.324,42.138,46.200,1.35
TR_000583385,2103-02-27 05:00:00,8.75,30.5,14.154,28.083,33.550,-8.74


'TR_000583385'

In [ ]:
# 환경변수 중에 집수면적(cat)과 유역면적(ws)도 하나의 이상치 점이 보임
# 1-1 EDA에서 환경변수는 한 관측소당 하나이므로 관측소 기준으로 groupby 시행

area_gb_df = df.groupby("station_id")[['cat_area', 'wsarea']].first().sort_values("cat_area", ascending = False)

import matplotlib.pyplot as plt

fig, ax = plt.subplots(
    1,  # 1은 생략가능
    1,  # 1은 생략가능
    figsize = (6, 4)
)

ax.scatter(area_gb_df['cat_area'], area_gb_df['wsarea'])
ax.set_xlabel('catchment area')
ax.set_ylabel('watershed area')

plt.tight_layout()
plt.show()

In [ ]:
# 환경변수 중에서 격자내 평균 하천차수가 혼자 7차인 경우가 있어서 확인필요
df.groupby("station_id")[['cat_area', 'wsarea', 'stream_order']].first().sort_values('stream_order', ascending = False)